In [1]:
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from dotenv import load_dotenv
import os
load_dotenv()
from langchain_classic.retrievers import EnsembleRetriever

c:\Users\palke\Desktop\Sample_Rag\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import subprocess
result = subprocess.run(['pip', 'show', 'langchain', 'langchain-community'], capture_output=True, text=True)
print(result.stdout)

Name: langchain
Version: 1.3.0
Summary: Building applications with LLMs through composability
Home-page: 
Author: 
Author-email: 
License: MIT
Location: C:\Users\palke\Desktop\Sample_Rag\myvenv\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-community
Version: 0.4.1
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
License: MIT
Location: C:\Users\palke\Desktop\Sample_Rag\myvenv\Lib\site-packages
Requires: aiohttp, dataclasses-json, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, PyYAML, requests, SQLAlchemy, tenacity
Required-by: 



In [3]:
chunks = [
    "Microsoft acquired GitHub for 7.5 billion dollars in 2018.",
    "Tesla Cybertruck production ramp begins in 2024.",
    "Google is a large technology company with global operations.",
    "Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.",
    "SpaceX develops Starship rockets for Mars missions.",
    "The tech giant acquired the code repository platform for software development.",
    "NVIDIA designs Starship architecture for their new GPUs.",
    "Tesla Tesla Tesla financial quarterly results improved significantly.",
    "Cybertruck reservations exceeded company expectations.",
    "Microsoft is a large technology company with global operations.", 
    "Apple announced new iPhone features for developers.",
    "The apple orchard harvest was excellent this year.",
    "Python programming language is widely used in AI.",
    "The python snake can grow up to 20 feet long.",
    "Java coffee beans are imported from Indonesia.", 
    "Java programming requires understanding of object-oriented concepts.",
    "Orange juice sales increased during winter months.",
    "Orange County reported new housing developments."

]

In [4]:
#convert to doc for langcahin
documents = [Document(page_content=chunk,metadata={"source": f"chunk_{i}"}) for i,chunk in enumerate(chunks)]

print('sample data:"')
for i,chunk in enumerate(chunks,1):
    print(f"{i}. {chunk}")

print("\n"+"="*80)

sample data:"
1. Microsoft acquired GitHub for 7.5 billion dollars in 2018.
2. Tesla Cybertruck production ramp begins in 2024.
3. Google is a large technology company with global operations.
4. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
5. SpaceX develops Starship rockets for Mars missions.
6. The tech giant acquired the code repository platform for software development.
7. NVIDIA designs Starship architecture for their new GPUs.
8. Tesla Tesla Tesla financial quarterly results improved significantly.
9. Cybertruck reservations exceeded company expectations.
10. Microsoft is a large technology company with global operations.
11. Apple announced new iPhone features for developers.
12. The apple orchard harvest was excellent this year.
13. Python programming language is widely used in AI.
14. The python snake can grow up to 20 feet long.
15. Java coffee beans are imported from Indonesia.
16. Java p

In [5]:
##CREATE THREE TYPES OF RETRIEVERS

#VECTOR(semantic/Dense) RETRIEVER
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12393.52it/s]


In [6]:
vector_retriever=vectorstore.as_retriever(search_kwargs={"k":2})

#test semantic retriever
query="What company acquired GitHub?"

results=vector_retriever.invoke(query)
for doc in results:
    print(f"Source: {doc.metadata['source']}, Content: {doc.page_content}")

Source: chunk_0, Content: Microsoft acquired GitHub for 7.5 billion dollars in 2018.
Source: chunk_5, Content: The tech giant acquired the code repository platform for software development.


In [ ]:
#2. BM25 (lexical) RETRIEVER
query="Tesla"
bm25_retriever=BM25Retriever.from_documents(documents)
bm25_retriever.k=3
#test bm25 retriever
results=bm25_retriever.invoke(query)
for doc in results:
    print(f"Source: {doc.metadata['source']}, Content: {doc.page_content}")


Source: chunk_7, Content: Tesla Tesla Tesla financial quarterly results improved significantly.
Source: chunk_3, Content: Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
Source: chunk_1, Content: Tesla Cybertruck production ramp begins in 2024.


In [ ]:
#3. HYBRID RETRIEVER


HybridRetriever = EnsembleRetriever(retrievers=[vector_retriever, bm25_retriever], weights=[0.5, 0.5])  

#test hybrid retriever
query="Query :What company acquired GitHub and what are the latest updates on Tesla's Cybertruck production?" 

results=HybridRetriever.invoke(query)



Source: chunk_3, Content: Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
Source: chunk_7, Content: Tesla Tesla Tesla financial quarterly results improved significantly.
Source: chunk_1, Content: Tesla Cybertruck production ramp begins in 2024.


In [16]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI

chunkypandey = []
def get_docs_for_query(query):
    results = HybridRetriever.invoke(query)
    return results
lol = get_docs_for_query(query)
for doc in lol:
    chunkypandey.append(doc)
    print(f"Source: {doc.metadata['source']}, Content: {doc.page_content}")
     
query = f"What company acquired GitHub and what are the latest updates on Tesla's Cybertruck production?  Context : {chunkypandey}"
llm = ChatOpenAI(
    model="openrouter/free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

system_message = SystemMessage(content="You are a helpful assistant that provides concise answers based on retrieved documents. When providing an answer, cite each document you base your answer on by their source name.")
human_message = HumanMessage(content=query,chunks=chunkypandey)

response =llm.invoke([system_message, human_message])
print("\n"+"="*80)
print("Answer from LLM:")
print(response.content)

Source: chunk_0, Content: Microsoft acquired GitHub for 7.5 billion dollars in 2018.
Source: chunk_1, Content: Tesla Cybertruck production ramp begins in 2024.
Source: chunk_8, Content: Cybertruck reservations exceeded company expectations.
Source: chunk_5, Content: The tech giant acquired the code repository platform for software development.

Answer from LLM:
- Microsoft acquired GitHub in 2018 for $7.5 billion【205bdd08-1a6c-4960-9724-283c21805edd】.  
- Tesla’s Cybertruck production is slated to begin ramping up in 2024【b16131d3-3778-4723-8c7e-e87fb63db160】.
